In [1]:
import numpy as np

In [2]:
import numpy as np

def butterfly(a, b, twiddle):
    """
    a, b: Liczby zespolone (wejścia)
    twiddle: Liczba zespolona (współczynnik W_N^k)
    """
    # Operacja motylkowa:
    # A = a + b*W
    # B = a - b*W
    temp = b * twiddle
    return a + temp, a - temp

def fft_radix2(x):
    N = len(x)
    if N <= 1: return x
    
    # Podział na parzyste i nieparzyste
    even = fft_radix2(x[0::2])
    odd = fft_radix2(x[1::2])
    
    # Współczynniki Twiddle
    T = [np.exp(-2j * np.pi * k / N) for k in range(N // 2)]
    
    # Łączenie (Butterfly)
    return [even[k] + T[k] * odd[k] for k in range(N // 2)] + \
           [even[k] - T[k] * odd[k] for k in range(N // 2)]

In [51]:
def fixed_point_butterfly(A_re, A_im, B_re, B_im, W_re, W_im):
    """
    Wszystkie argumenty wejściowe muszą być liczbami całkowitymi (int)
    reprezentującymi ułamki w formacie Q1.15.
    """
    
    # ----------------------------------------------------
    # KROK 1: Mnożenie zespolone (B * W)
    # W FPGA mnożenie dwóch liczb 16-bitowych daje wynik 32-bitowy (format Q2.30)
    # ----------------------------------------------------
    temp_re = (B_re * W_re) - (B_im * W_im)
    temp_im = (B_re * W_im) + (B_im * W_re)
    
    # ----------------------------------------------------
    # KROK 2: Skalowanie (Bit-shift)
    # Przesunięcie bitowe o 15 w prawo ucina "nadmiarowe" bity ułamkowe 
    # i przywraca wynik mnożenia do bazowego formatu Q1.15
    # ----------------------------------------------------
    temp_re = temp_re >> 15
    temp_im = temp_im >> 15
    
    # ----------------------------------------------------
    # KROK 3: Operacja sumowania i odejmowania (Butterfly)
    # ----------------------------------------------------
    out_A_re = A_re + temp_re
    out_A_im = A_im + temp_im
    
    out_B_re = A_re - temp_re
    out_B_im = A_im - temp_im
    
    # W rygorystycznym modelu hardware'owym tutaj następuje ew. obcięcie do 16-bit
    # np. ograniczenie wyników do zakresu [-32768, 32767] (Saturacja)
    
    return out_A_re, out_A_im, out_B_re, out_B_im

In [52]:
# Funkcje pomocnicze do konwersji
def float_to_q15(val):
    # Skalowanie i obcięcie do zakresu int16
    return int(np.clip(val * 32768.0, -32768, 32767))

def q15_to_float(val):
    return val / 32768.0

# 1. Dane wejściowe w float (tak jak czytasz z mikrofonu)
a_re_f, a_im_f = 0.5, 0.1
b_re_f, b_im_f = -0.3, 0.4
w_re_f, w_im_f = 0.707, -0.707 # Zbliżenie np. exp(-j*pi/4)

# 2. Konwersja na liczby całkowite dla FPGA
A_re = float_to_q15(a_re_f)
A_im = float_to_q15(a_im_f)
B_re = float_to_q15(b_re_f)
B_im = float_to_q15(b_im_f)
W_re = float_to_q15(w_re_f)
W_im = float_to_q15(w_im_f)

# 3. Wykonanie symulacji sprzętowej (tylko int i przesunięcia!)
out_A_re, out_A_im, out_B_re, out_B_im = fixed_point_butterfly(A_re, A_im, B_re, B_im, W_re, W_im)

# 4. Porównanie z "idealną" matematyką zmiennoprzecinkową
A_ideal = complex(a_re_f, a_im_f) + complex(b_re_f, b_im_f) * complex(w_re_f, w_im_f)
B_ideal = complex(a_re_f, a_im_f) - complex(b_re_f, b_im_f) * complex(w_re_f, w_im_f)

print("--- Wynik Hardware (Q1.15 -> Float) ---")
print(f"A' = {q15_to_float(out_A_re):.4f} + {q15_to_float(out_A_im):.4f}j")
print(f"B' = {q15_to_float(out_B_re):.4f} + {q15_to_float(out_B_im):.4f}j")

print("\n--- Wynik Idealny (Float64) ---")
print(f"A' = {A_ideal.real:.4f} + {A_ideal.imag:.4f}j")
print(f"B' = {B_ideal.real:.4f} + {B_ideal.imag:.4f}j")

--- Wynik Hardware (Q1.15 -> Float) ---
A' = 0.5707 + 0.5948j
B' = 0.4293 + -0.3949j

--- Wynik Idealny (Float64) ---
A' = 0.5707 + 0.5949j
B' = 0.4293 + -0.3949j


In [54]:
import math

def generate_fsm_golden_table(N):
    num_stages = int(math.log2(N))
    print(f"=== ZŁOTY MODEL ADRESOWANIA FFT DLA N = {N} ===\n")
    
    # Zewnętrzna pętla: Etapy (Stages)
    for stage in range(1, num_stages + 1):
        print(f"--- ETAP {stage} ---")
        
        # 'half_step' to odległość w pamięci między próbką A i B w danym etapie
        half_step = 1 << (stage - 1) 
        
        # 'step' to całkowity rozmiar grupy motylków
        step = 1 << stage 
        
        # Środkowa pętla: Skoki po kolejnych grupach motylków
        for group_start in range(0, N, step):
            
            # Wewnętrzna pętla: Kolejne motylki wewnątrz jednej grupy
            for b_idx in range(half_step):
                
                # Wyliczanie adresów dla pamięci danych (BRAM)
                index_A = group_start + b_idx
                index_B = index_A + half_step
                
                # Wyliczanie adresu (indeksu k) dla pamięci ROM (Twiddle)
                # Formuła: k = b_idx * (N / step)
                k = b_idx * (N // step)
                
                print(f"Czytaj z BRAM -> A: {index_A:2d}, B: {index_B:2d} | Czytaj z ROM -> Twiddle: W^{k}")
        print("")

# Uruchomienie dla N=8

In [58]:
generate_fsm_golden_table(8)


=== ZŁOTY MODEL ADRESOWANIA FFT DLA N = 8 ===

--- ETAP 1 ---
Czytaj z BRAM -> A:  0, B:  1 | Czytaj z ROM -> Twiddle: W^0
Czytaj z BRAM -> A:  2, B:  3 | Czytaj z ROM -> Twiddle: W^0
Czytaj z BRAM -> A:  4, B:  5 | Czytaj z ROM -> Twiddle: W^0
Czytaj z BRAM -> A:  6, B:  7 | Czytaj z ROM -> Twiddle: W^0

--- ETAP 2 ---
Czytaj z BRAM -> A:  0, B:  2 | Czytaj z ROM -> Twiddle: W^0
Czytaj z BRAM -> A:  1, B:  3 | Czytaj z ROM -> Twiddle: W^2
Czytaj z BRAM -> A:  4, B:  6 | Czytaj z ROM -> Twiddle: W^0
Czytaj z BRAM -> A:  5, B:  7 | Czytaj z ROM -> Twiddle: W^2

--- ETAP 3 ---
Czytaj z BRAM -> A:  0, B:  4 | Czytaj z ROM -> Twiddle: W^0
Czytaj z BRAM -> A:  1, B:  5 | Czytaj z ROM -> Twiddle: W^1
Czytaj z BRAM -> A:  2, B:  6 | Czytaj z ROM -> Twiddle: W^2
Czytaj z BRAM -> A:  3, B:  7 | Czytaj z ROM -> Twiddle: W^3



# Architektura Sprzętowa: Radix-2 FFT IP Core

Poniższy dokument opisuje przepływ sterowania (FSM) oraz przepływ danych (Data Path) dla sprzętowej implementacji algorytmu Szybkiej Transformaty Fouriera w języku Verilog.

## 1. Maszyna Stanów (Control Path - FSM)

Moduł: `fft_address_generator`
Zadanie: Zarządzanie licznikami pętli i wyliczanie adresów dla pamięci BRAM (dane) oraz ROM (współczynniki).

```mermaid
stateDiagram-v2
    [*] --> IDLE : Reset (rst_n = 0)
    
    IDLE --> CALC : start = 1\n(Inicjalizacja liczników:\nstage=1, group=0, bfly=0)
    
    state CALC {
        [*] --> BFLY_OP
        
        BFLY_OP : Oblicz Adresy A, B, W
        BFLY_OP : addr_A = group_cnt + bfly_cnt
        BFLY_OP : addr_B = addr_A + half_step
        BFLY_OP : addr_W = bfly_cnt << (LOG2_N - stage_cnt)
        
        BFLY_OP --> INC_BFLY
        
        INC_BFLY : bfly_cnt++
        INC_BFLY --> BFLY_OP : bfly_cnt < half_step
        
        INC_BFLY --> INC_GROUP : bfly_cnt == half_step
        
        INC_GROUP : bfly_cnt = 0
        INC_GROUP : group_cnt += step
        INC_GROUP --> BFLY_OP : group_cnt < N_POINTS
        
        INC_GROUP --> INC_STAGE : group_cnt == N_POINTS
        
        INC_STAGE : group_cnt = 0
        INC_STAGE : stage_cnt++
        INC_STAGE --> BFLY_OP : stage_cnt <= LOG2_N
    }
    
    CALC --> DONE : stage_cnt > LOG2_N
    
    DONE : Ustaw flagę `done = 1`
    DONE --> IDLE : Automatyczny powrót\n(w następnym takcie)